# A3 — CIFAR-10 validation submission starter

This notebook prepares the fixed data split and exports your predictions. Implement your own model in the marked section. It contains no TA model implementation or trained weights.

- Colab: unzip `A3_Kaggle_student_files.zip` into the notebook working directory first.
- Kaggle: attach this competition's data. The cell below also searches `/kaggle/input`.
- Part II uses NumPy for the model and gradients; torchvision below is only the dataset loader named in the handout.
- Kaggle uses **only** original training indices 45000–49999. Keep the official 10,000-image test set for final evaluation after model selection.
- This is a practice leaderboard on a known validation set; it is not a hidden test evaluation.

In [ ]:
from pathlib import Path
import random
import sys
import numpy as np

random.seed(0)
np.random.seed(0)
candidates = [Path('.'), Path('student_files')]
if Path('/kaggle/input').exists():
    candidates += [p.parent for p in Path('/kaggle/input').rglob('validation_images.npz')]
DATA_DIR = next((p for p in candidates if (p / 'validation_images.npz').is_file()), None)
if DATA_DIR is None:
    raise FileNotFoundError('Unzip the student files or attach the competition data first.')
sys.path.insert(0, str(DATA_DIR.resolve()))
from a3_submission import export_submission
bundle = np.load(DATA_DIR / 'validation_images.npz', allow_pickle=False)
validation_ids = bundle['id']
validation_images = bundle['images']
assert set(bundle.files) == {'id', 'images'}
assert np.array_equal(validation_ids, np.arange(45000, 50000))
assert validation_images.shape == (5000, 32, 32, 3)
print('Validation images:', validation_images.shape, validation_images.dtype)
print('IDs:', validation_ids[0], 'through', validation_ids[-1])

## Load the official training split

This downloads CIFAR-10 when it is not already cached. Internet access is needed for the first download. Split **before shuffling**. The training mean below uses only the first 45,000 images. The validation labels are available for local model selection, but must not be used as training targets or copied into submissions.

In [ ]:
from torchvision.datasets import CIFAR10

CIFAR_ROOT = Path('./cifar10_data')
dataset = CIFAR10(root=str(CIFAR_ROOT), train=True, download=True)
assert dataset.data.shape == (50000, 32, 32, 3)
assert np.array_equal(dataset.data[45000:], validation_images)
all_train = dataset.data.astype(np.float32) / np.float32(255.0)
mean_image = all_train[:45000].mean(axis=0, dtype=np.float64).astype(np.float32)
all_train -= mean_image
X_train_hwc = all_train[:45000]
X_validation_hwc = all_train[45000:]
y_train = np.asarray(dataset.targets[:45000], dtype=np.int64)
y_validation = np.asarray(dataset.targets[45000:], dtype=np.int64)
# Use these arrays for the NumPy model.
X_train_flat = X_train_hwc.reshape(45000, 3072)
X_validation_flat = X_validation_hwc.reshape(5000, 3072)
# CNN input order is N,C,H,W; keep the same split.
X_train_chw = X_train_hwc.transpose(0, 3, 1, 2)
X_validation_chw = X_validation_hwc.transpose(0, 3, 1, 2)
print('Train:', X_train_flat.shape, 'Validation:', X_validation_flat.shape)

## Your model

Implement and train your own model, then set `predictions` to a length-5,000 integer array in the order of `validation_ids`. If your output is logits of shape `(5000, 10)`, use `logits.argmax(axis=1)`.

Use seeds 0. For Part III also set the PyTorch RNG seed(s) to 0 before creating the model and data loaders. Do not use pretrained weights or external data. Follow the handout for all experiments and the e3 submission.

In [ ]:
# TODO: implement and train your model here.
# predictions = ...  # Your model's class predictions; do not assign y_validation.
predictions = None

In [ ]:
if predictions is None:
    print('Add your model predictions above, then rerun this cell.')
else:
    output = export_submission(predictions, validation_ids, 'submission.csv')
    print('Saved', output.resolve())
    print('Local validation accuracy:', float(np.mean(np.asarray(predictions) == y_validation)))
    print('Upload submission.csv to the competition. State Part II or Part III in its description.')

## Format-only check (optional)

The all-zero file below checks CSV formatting only. It is **not** a trained model or TA baseline, and should not be used as your assignment submission.

In [ ]:
format_only = export_submission(np.zeros(5000, dtype=np.int64), validation_ids, 'format_check_only.csv')
print(format_only.resolve())